# Quarry likelihood from embedding similarity

Score every 10 m pixel in Fiji by whether its [OlmoEarth v1.3 embedding](https://source.coop/olmoearth/olmoearth-v1-3-embeddings) is more like the quarry training points or the non-quarry ones.

1. Sample the embedding at every training point: quarry and non-quarry.
2. For every pixel: `margin = best cosine similarity to a quarry point - best to a non-quarry point`.
3. Squash the margin to a single 0-100 **quarry likelihood**: 50 = equally like both, 100 = clearly quarry.
4. Map pixels at or above a threshold over satellite imagery.

In [ ]:
import numpy as np
import xarray as xr
from dask.distributed import Client as DaskClient

from utils import BBOX, load_embeddings, sample_embeddings, training_data

In [ ]:
# BBOX from utils. West of 180° Fiji is all UTM zone 60; east of 180° (Lau group) is zone 1, not done here.
YEAR = 2024  # OlmoEarth is global for 2023-2025
QUARRY = 8  # lulc_code
CHUNKS = {"x": 1024, "y": 1024}  # 1024 x 1024 x 64 float32 = 268 MB per block
STRIP = 64  # rows scored at once: (STRIP x 1024 x n_points) float32 per thread
TEMPERATURE = 0.01  # margin -> likelihood steepness: margin 0.01 -> 73, 0.02 -> 88

In [ ]:
dask_client = DaskClient(n_workers=1, threads_per_worker=16, memory_limit="16GB")

## Embeddings

Lazy; only bands 0-63 (a complete 64-d Matryoshka embedding). `utils.load_embeddings` picks the UTM zone from the bbox.

In [ ]:
emb = load_embeddings(BBOX, YEAR, chunks=CHUNKS)
emb

In [ ]:
def normalise(q: np.ndarray) -> np.ndarray:
    """Quantized embeddings (..., band), NaN = nodata -> unit vectors (NaN stays NaN)."""
    v = (np.abs(q) / 127.5) ** 2 * np.sign(q)  # signed-power de-quantization
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

## Training point embeddings

In [ ]:
samples = sample_embeddings(training_data, YEAR)  # all points, ~4 min
vectors = normalise(samples.values)
valid = ~np.isnan(vectors).any(axis=1)
is_quarry = (training_data["lulc_code"] == QUARRY).values
quarry = vectors[valid & is_quarry]
other = vectors[valid & ~is_quarry]
points = training_data[is_quarry]
print(f"{len(quarry)} quarry and {len(other)} non-quarry points with embeddings")

In [ ]:
def likelihood(v: np.ndarray, exclude_self: bool = False) -> np.ndarray:
    """Unit vectors (n, band) -> quarry likelihood 0-1 (NaN stays NaN)."""
    sq, so = v @ quarry.T, v @ other.T
    if exclude_self:  # v is the training points: don't let a point match itself
        sq[sq > 0.99999], so[so > 0.99999] = -1, -1
    margin = sq.max(axis=-1) - so.max(axis=-1)
    return 1 / (1 + np.exp(-margin / TEMPERATURE))


# Check on the training points themselves (each scored against the others).
# Nearby points from the same site make this optimistic.
p = likelihood(vectors[valid], exclude_self=True) * 100
for name, mask in [("quarry", is_quarry[valid]), ("non-quarry", ~is_quarry[valid])]:
    print(
        f"{name:>10}: median {np.median(p[mask]):.0f}, >= 50: {(p[mask] >= 50).mean():.0%}"
    )

## Score every pixel

In [ ]:
def quarry_likelihood(block: np.ndarray) -> np.ndarray:
    """(y, x, band) embeddings -> (y, x) uint8 quarry likelihood 0-100; 255 = nodata.

    Scored in STRIP-row strips to cap memory; all-nodata (ocean) strips are skipped.
    """
    out = np.full(block.shape[:-1], 255, "uint8")
    for i in range(0, len(block), STRIP):
        b = block[i : i + STRIP]
        if np.isnan(b).all():
            continue
        p = likelihood(normalise(b))
        out[i : i + STRIP] = np.where(np.isnan(p), 255, np.round(p * 100))
    return out


score = xr.apply_ufunc(
    quarry_likelihood,
    emb.to_dataarray("band").chunk(band=-1),
    input_core_dims=[["band"]],
    dask="parallelized",
    output_dtypes=["uint8"],
)
score.attrs["nodata"] = 255
score

In [ ]:
THRESHOLD = 50  # quarry likelihood %: 50 = more like quarry than non-quarry

In [ ]:
# Write the results to a COG
from odc.geo.xr import write_cog

write_cog(
    score,
    "score.tif",
    overwrite=True,
)

## Map

In [ ]:
m = score.where((score >= THRESHOLD) & (score <= 100)).odc.explore(
    cmap="Spectral",
    vmin=THRESHOLD,
    vmax=100,
    tiles="Esri.WorldImagery",
    name=f">= {THRESHOLD}% quarry likelihood",
)
points.explore(m=m, color="cyan", name="quarry training points")
m